# WildSpeech M8.7 — Child Speech Intelligence, Full ASR + A00–A16 Experiments

**New research master notebook (unexecuted until your private audio/GPU run).**

This notebook orchestrates: audio inventory; pinned Pasketti-phonetic FP32 inference with WavLM hidden layers; VAD/audio quality; target IPA; task-aware child-response localization; whole/segment/random CTC features; optional Pasketti Word; grouped item-correctness models; item-normalized student aggregation; A00–A16 frozen-fold outcome comparisons; training-only QA; negative controls; and full scientific plots.

**Frozen comparators:** M8.4 original reference, M8.6 I5 student OOF R² 0.2370 and M8.6 I4 school OOF R² 0.1706. These figures are historical, not new experiment results.

**Data safety:** no test recording may be manually listened to or transcribed; no online inference services; no model adaptation from test data; all student and speech artifacts stored privately on Google Drive.

**Execution modes:** start with a GPU smoke subset, finish the complete 24,518-clip cache, then use SCREEN and finally FULL (100 original outer folds, nested learning). No full score is claimed without that final run.

## 0. Mount private Drive and choose the GPU runtime

Choose a Colab NVIDIA GPU when extracting Pasketti. Full run decoding is not appropriate for a CPU-only session. Pasketti Word uses a separate vLLM-compatible Linux environment, covered below.

In [ ]:
import os,sys,json,subprocess,importlib,time
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display,Markdown,Image
try:
    ON_COLAB=importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:ON_COLAB=False
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive",force_remount=False)
    DRIVE=Path("/content/drive/MyDrive")
else:
    DRIVE=Path(os.environ.get("WILDSPEECH_DRIVE_ROOT",str(Path.home()/"MyDrive")))
assert DRIVE.is_dir(),f"Drive unavailable: {DRIVE}"
print("Private Drive:",DRIVE)

## 1. Install version-pinned M8.7 source

The phonetic model card verified transformers 4.57.6 and FP32 WavLM, and expects mono 16 kHz audio. Model weights will be fetched once from Hugging Face; **no competition audio is uploaded**. Use a separate runtime for the optional 1.7B word model.

In [ ]:
URL="https://github.com/mybenkhadda/WildSpeech.git"
CHECKOUT=Path("/content/WildSpeech_git") if ON_COLAB else Path.cwd()/"WildSpeech_git"
if not (CHECKOUT/".git").is_dir():
    if CHECKOUT.exists():raise RuntimeError(f"Existing non-Git directory: {CHECKOUT}")
    subprocess.run(["git","clone","--depth","1","--branch","main",URL,str(CHECKOUT)],check=True)
else:
    subprocess.run(["git","-C",str(CHECKOUT),"pull","--ff-only","origin","main"],check=True)
PKG=CHECKOUT/"WildSpeech_M8"
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(PKG)+"[m87]"],check=True)
source=str(PKG/"src")
if source not in sys.path:sys.path.insert(0,source)
GIT_SHA=subprocess.check_output(["git","-C",str(CHECKOUT),"rev-parse","HEAD"],text=True).strip()
os.environ["WILDSPEECH_COMMIT"]=GIT_SHA
print("Source commit:",GIT_SHA)
import torch
print("Torch:",torch.__version__,"CUDA:",torch.cuda.is_available())
if torch.cuda.is_available():print("GPU:",torch.cuda.get_device_name(0))

## 2. Set paths and execution switches

If the audio archives are not in an obvious location, set TRAIN_TAR and TEST_TAR explicitly. Both must be **uncompressed seekable .tar files**, or provide extracted private audio directories. The scripts index tar members without extracting all recordings.

Set LICENSE_ACKNOWLEDGED only after reviewing the checkpoint license and official competition eligibility. The checkpoint's model card currently identifies bigcode-openrail-m; this notebook does not declare that license automatically eligible.

In [ ]:
from wildspeech_m87.contracts import Config,ASSETS,EXPERIMENTS,FULL_FOLD_HASH
from wildspeech_m84.data import locate_data_root,load_official
DATA_ROOT=locate_data_root(DRIVE)
PHASE0_ROOT=DRIVE/"WildSpeech_M84"/"phase0"
M86_ROOT=DRIVE/"WildSpeech_M86"/"m0pp"/"full"
OUTPUT=DRIVE/"WildSpeech_M87"/"speech"
TRAIN_TAR=""                       # optional: absolute path to train .tar
TEST_TAR=""                        # optional: absolute path to test .tar
TRAIN_AUDIO_DIR=""                 # alternative: private directory containing train audio
TEST_AUDIO_DIR=""                  # alternative: private directory containing test audio
LICENSE_ACKNOWLEDGED=False        # change only after explicit license/rules review
RUN_GPU_EXTRACT=False             # True to run full CTC extraction
RUN_WORD_ASR=False               # separate supported vLLM Linux runtime
RUN_EVALUATION=False             # True once complete cache exists
MODE="SCREEN"                     # change to FULL after SCREEN figures and QA
MAX_CLIPS=0                       # 0=all; use 64 for initial smoke only
VAD_METHOD="silero"               # energy is explicitly a test-only fallback
BATCH_SIZE=2
print("Audio settings:",TRAIN_TAR,TEST_TAR,TRAIN_AUDIO_DIR,TEST_AUDIO_DIR)
display(pd.DataFrame([{"asset":a,**v} for a,v in ASSETS.items()]))

## 3. Verify data and fixed outer folds

Read original 541 training students, 361 test students, 14,704 training clips and 9,814 test clips. Read exact Phase 0 folds and M8.6 I5/I4 OOF, not new splits.

In [ ]:
from wildspeech_m87.contracts import validate_baselines,clip_inventory
DATA=load_official(DATA_ROOT)
FOLDS,REFERENCE,INVENTORY,ANCHORS=validate_baselines(
    DATA,PHASE0_ROOT,M86_ROOT,official=True)
display(pd.DataFrame([ANCHORS]).T)
assert ANCHORS["fold_hash"]==FULL_FOLD_HASH
print("Folds:",FOLDS.groupby(["scheme","repeat"]).size().to_dict())
print("Train/test audio inventory:",INVENTORY.groupby(["split","task"]).size().to_dict())

## 4. Locate audio archives (or set explicit directory paths)

The folder audit below searches only bounded likely locations and does not scan the entire Drive recursively. Set paths manually if ambiguous.

In [ ]:
possible=[DATA_ROOT,DATA_ROOT.parent,DATA_ROOT/"audio",
          DRIVE/"WildSpeech",DRIVE/"WildSpeech"/"audio",DRIVE/"WildSpeech_Data"]
for folder in dict.fromkeys(possible):
    if folder.is_dir():
        print("\n",folder)
        print([x.name for x in list(folder.iterdir())[:70]
               if x.suffix.lower() in (".tar",".flac",".wav") or "audio" in x.name.lower()])

## 5. Freeze configuration and inspect provenance

In [ ]:
CONFIG=Config(drive_root=str(DRIVE),data_root=str(DATA_ROOT),
    output_root=str(OUTPUT),train_tar=TRAIN_TAR,test_tar=TEST_TAR,
    train_audio_dir=TRAIN_AUDIO_DIR,test_audio_dir=TEST_AUDIO_DIR,
    acknowledge_license_review=LICENSE_ACKNOWLEDGED,
    batch_size=BATCH_SIZE,mode=MODE)
print("Private output:",CONFIG.output())
print("N models:",len(EXPERIMENTS),"| Source SHA:",GIT_SHA)
print("No automatically assumed model eligibility; review license before running inference.")

## 6. Run Pasketti-phonetic CTC, layers 6–10, Silero VAD and quality extraction

The model stays in FP32; frame probabilities are saved in compressed float16 shards only after stable inference. Each clip gets a unique ID and status VALID or FAILED. Completed shards resume without rereading audio. Use MAX_CLIPS=64 for the pilot, then reset to zero for the full inventory.

In [ ]:
from wildspeech_m87.gpu_cache import extract,summarize_cache
if RUN_GPU_EXTRACT:
    assert LICENSE_ACKNOWLEDGED
    assert TRAIN_TAR or TRAIN_AUDIO_DIR, "Set train TAR/directory"
    assert TEST_TAR or TEST_AUDIO_DIR, "Set test TAR/directory"
    coverage=extract(CONFIG,INVENTORY,
        max_clips=MAX_CLIPS,shard_size=48,
        vad_method=VAD_METHOD)
    print("Extraction coverage:",coverage)
else:
    print("GPU extraction disabled. Set RUN_GPU_EXTRACT=True once audio sources are configured.")

## 7. Verify complete clip coverage and extracted feature quality

In [ ]:
from wildspeech_m87.gpu_cache import load_features
CACHE_FEATURES=None
if (OUTPUT/"features"/"phonetic_clip_features_PRIVATE.parquet").exists():
    CACHE_FEATURES=load_features(OUTPUT)
    display(CACHE_FEATURES.status.value_counts(dropna=False).to_frame())
    print("Unique clip IDs:",CACHE_FEATURES.uid.nunique())
    display(CACHE_FEATURES.groupby(["split","task","status"]).size().to_frame("n"))
else:print("No real cache found — do not interpret missing results as zero performance.")

## 8. C3–C6 evidence figures: coverage, timing, uncertainty, correctness strata

Generate figures ONLY from the actual cached feature table. All artifacts stay on Drive.

In [ ]:
from wildspeech_m87.diagnostics import extraction_atlas
if CACHE_FEATURES is not None:
    ATLAS=extraction_atlas(OUTPUT,INVENTORY,CACHE_FEATURES)
    print("Evidence atlas:",ATLAS)
    for fn in ("01_cache_coverage.png","02_duration_by_task.png",
               "03_localization_confidence.png","04_whole_vs_segment_phone_edits.png",
               "05_ctc_segment_delta.png","06_grading_within_correctness.png"):
        path=OUTPUT/"figures"/fn
        if path.exists():display(Image(filename=str(path)))
else:print("Atlas skipped until extraction cache exists.")

## 9. Phonetic target mapping and explicit failures

Inspect unknown G2P mappings and IPA tokenizer coverage. No forced alignment should silently invent unsupported phonemes.

In [ ]:
targets=OUTPUT/"manifests"/"target_registry_PRIVATE.parquet"
if targets.exists():
    t=pd.read_parquet(targets)
    display(t.groupby(["task","reason"]).size().to_frame("n"))
    print("Target-valid share:",float(t.valid.mean()))
    display(t[~t.valid][["task","expected_text","reason"]].head(30))
else:print("Run phonetic extraction to generate target registry.")

## 10. Training-only waveform localization QA

The plots and manual QA must use training clips ONLY. Test recordings must not be listened to or manually transcribed.

In [ ]:
from wildspeech_m87.localization_qa import create_template,evaluate_qa
QA_FILE=OUTPUT/"qa"/"training_qa_template_PRIVATE.csv"
if CACHE_FEATURES is not None and not QA_FILE.exists():
    qa=create_template(DATA,INVENTORY,CACHE_FEATURES,QA_FILE,n=150)
    print("Training QA template:",QA_FILE,"rows:",len(qa))
elif QA_FILE.exists():
    print("Existing QA annotations preserved:",QA_FILE)
if QA_FILE.exists():
    result=evaluate_qa(QA_FILE,OUTPUT/"qa")
    print("QA status:",result)
    if result.get("status")=="NOT_REVIEWED":
        print("No reviewed annotations. Localization ≥80% has NOT been established.")

## 11. Optional training-only waveform figures

Create the 12 example waveform plots using the training archive only; they show the predicted response region and support manual review.

In [ ]:
MAKE_TRAINING_QA_WAVEFORMS=False
if MAKE_TRAINING_QA_WAVEFORMS and CACHE_FEATURES is not None:
    from wildspeech_m87.gpu_cache import create_sources
    from wildspeech_m87.diagnostics import qa_training_waveforms
    s=create_sources(CONFIG,OUTPUT)["train"]
    n=qa_training_waveforms(OUTPUT,INVENTORY,CACHE_FEATURES,s,n=12)
    print("Training QA figures:",n)

## 12. Optional local Pasketti Word inference for sentence repetition

This model uses Qwen3-ASR-1.7B with qwen_asr[vllm] 0.0.6, typically requiring a separate compatible Linux/CUDA runtime. Do not run both heavy Pasketti models simultaneously. If absent, A07 is reported NOT_ELIGIBLE; it is never assigned a fake score.

In [ ]:
if RUN_WORD_ASR:
    assert LICENSE_ACKNOWLEDGED
    from wildspeech_m87.gpu_cache import extract_words
    result=extract_words(CONFIG,INVENTORY,max_clips=MAX_CLIPS,
                         sentence_only=True)
    print("Word ASR cached:",result)
else:print("Pasketti Word disabled — A07 will be marked NOT_ELIGIBLE until generated.")

## 13. Task-specific phonetic and word partial-credit descriptions

Inspect real correlations with observed item_score, emphasizing *graded scores within each correctness stratum*, rather than reporting ASR accuracy as future-reading prediction.

In [ ]:
if (OUTPUT/"features"/"phonetic_clip_features_PRIVATE.parquet").exists():
    c=load_features(OUTPUT)
    fields=[x for x in ("child_phone_edit_rate","whole_phone_edit_rate",
          "task_phone_graded_accuracy","child_target_logp_per_phone",
          "word_coverage","word_lcs_ratio","clip_pred_correct") if x in c]
    if fields:
        display(c.groupby("task")[fields].agg(["mean","std","count"]).round(4))
else:print("Task plots require real cached phonetic results.")

## 14. Predeclared A00–A16 experiment registry

Includes whole vs child target scoring, entropy, nuisance-only controls, random-segment null, within-correctness, task-specific partial credit, grouped clip correctness, unscored recovery, confidence weighting, layer sketches, residual arm, and proper inner-OOF stacking.

In [ ]:
from wildspeech_m87.evaluation import experiment_plan
PLAN=pd.DataFrame(experiment_plan())
display(PLAN[["comparison_id","candidate","model","negative_control",
              "main_hypothesis"]])
assert len(PLAN)==17
print("Predeclared primary hypotheses:",PLAN.loc[PLAN.main_hypothesis,"candidate"].tolist())

## 15. Run A00–A16 matched-fold evaluation

SCREEN: first 5 student + 5 school folds; full model outcomes are not confirmatory. FULL: 50 student and 50 school-disjoint folds with 3 nested student folds. All model training uses only outer-training labels. Completed fold checkpoints resume automatically.

In [ ]:
if RUN_EVALUATION:
    assert MODE in ("SCREEN","FULL")
    from wildspeech_m87.evaluation import evaluate
    RESULTS=evaluate(drive_root=DRIVE,data_root=str(DATA_ROOT),
        cache_root=str(OUTPUT),phase0_root=str(PHASE0_ROOT),
        m86_root=str(M86_ROOT),mode=MODE)
    print("Decision:",RESULTS["decision"],"winner:",RESULTS["winner"])
    print("Skipped arms:",RESULTS["skipped"])
else:print("Enable RUN_EVALUATION after complete audio cache and QA review.")

## 16. Models ranked by student and school R²

Keep original M8.6 I5 and I4 metrics in the same table. A4/A5/A15 are NEGATIVE CONTROLS, not promotion candidates.

In [ ]:
EVAL_DIR=OUTPUT/"evaluations"/MODE.lower()
RESULT_JSON=EVAL_DIR/"reports"/"M87_RESULTS.json"
if RESULT_JSON.exists():
    RESULTS=json.loads(RESULT_JSON.read_text())
    table=pd.DataFrame(RESULTS["model_metrics"])
    show=[c for c in ("model","student_pooled_r2","school_pooled_r2",
                       "student_mae","student_rmse","mean_fold_delta_r2",
                       "win_fraction","corrected_t","engineering_eligible",
                       "science_gate","status","reason") if c in table]
    display(table[show].round(5))
else:print("No outcome R² results yet.")

## 17. Frozen-fold diagnostic plots and transfer frontier

In [ ]:
for filename in ("07_model_student_school_r2.png",
                 "08_paired_delta_r2.png","09_transfer_frontier.png",
                 "10_baseline_prediction_scatter.png"):
    path=EVAL_DIR/"figures"/filename
    if path.exists():
        print(filename)
        display(Image(filename=str(path)))

## 18. Paired R² against I5, instrument vetoes and robustness

Report win rate, corrected Nadeau–Bengio statistic, school pooled ΔR² and per-instrument R² changes. The corrected statistic is approximate, particularly after exploring many model families.

In [ ]:
paired=EVAL_DIR/"tables"/"07_paired_science_gate.csv"
if paired.exists():
    p=pd.read_csv(paired)
    display(p.round(6))
    print("No post-hoc changes to preregistered statistical thresholds.")
else:print("Paired comparison table requires FULL execution.")

## 19. Training-only outcome residual and school transfer atlas

Observe whether child speech features predict future reading residuals left by I5. The student embedding used for predictive claims must be generated out-of-fold.

In [ ]:
for filename in ("05_student_oof_residuals_PRIVATE.csv",
                 "06_within_correctness_PRIVATE.csv"):
    p=EVAL_DIR/"tables"/filename
    if p.exists():
        print(filename)
        display(pd.read_csv(p).head(15))

## 20. Perturbation-stability study (TRAIN clips only)

Optional: reread up to 45 TRAIN recordings and recompute phonetic evidence under 0.9× and 1.1× speed changes and added noise. These are stress tests; do not claim invariance is guaranteed.

In [ ]:
RUN_STABILITY=False
if RUN_STABILITY:
    assert LICENSE_ACKNOWLEDGED
    from wildspeech_m87.phonetic_asr import PhoneticASR
    from wildspeech_m87.gpu_cache import create_sources
    from wildspeech_m87.targets import TargetRegistry,normalize_text
    from wildspeech_m87.stability import run_stability
    model=PhoneticASR(layers=(6,),max_seconds=30.)
    source=create_sources(CONFIG,OUTPUT)["train"]
    registry=TargetRegistry(model.processor.tokenizer).make(INVENTORY[INVENTORY.split=="train"])
    mapping={(r.task,normalize_text(r.expected_text)):r.ids
             for r in registry.itertuples(index=False) if r.valid}
    STABILITY=run_stability(INVENTORY,source,model,mapping,
                           OUTPUT/"stability",max_clips=45)
    print(STABILITY)

## 21. Statistical decision and detailed science report

If the child localization QA has not been reviewed and passed, the output is *experimental evidence*, not a verified child-specific measurement.

In [ ]:
REPORT=EVAL_DIR/"reports"/"M87_DECISION.md"
if REPORT.exists():display(Markdown(REPORT.read_text(encoding="utf-8")))
else:print("No FULL science report until A00–A16 experiments have executed.")

## 22. Conditional final submission exporter — no automatic online upload

Requires FULL engineering gate and reviewed localization accuracy/false adult-selection checks. Independently verifies test singletons and row permutations; produces a private CSV/checksum only.

In [ ]:
RUN_SUBMISSION_EXPORT=False
if RUN_SUBMISSION_EXPORT:
    from wildspeech_m87.submission import export_if_eligible
    manifest=export_if_eligible(
        drive_root=DRIVE,cache_root=OUTPUT,
        eval_root=OUTPUT/"evaluations"/"full",
        data_root=str(DATA_ROOT),rasch_iters=7,require_qa=True)
    print("Verified candidate:",manifest["file"],"SHA:",manifest["sha256"])
else:print("Submission not exported or submitted online.")

## 23. Private artifact inventory

In [ ]:
for folder in ("manifests","indexes","shards","features","figures","qa",
               "evaluations/screen","evaluations/full","stability"):
    path=OUTPUT/folder
    if path.is_dir():
        files=list(path.iterdir())
        print(folder,":",len(files),"entries",
              [f.name for f in files[:5]])
    else:print(folder,": not generated")

## 24. Handoff to M8.8

If A02 (child-only evidence) improves over A01 (whole-clip evidence), and the QA gate is satisfied, prioritize sentence-repetition partial credit and task-specific phonetic error topology.

If no speech family improves I5/I4, preserve the screening baselines and investigate localization, ASR reliability, item alignment and recording nuisances first. Do not train another large end-to-end student Transformer based on a null result.

No results in this notebook are fabricated: all tables and plots are conditional on actual private data extraction and completed experiments.